##  Stockout Risk Prediction

## Import Libraries

In [129]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split,StratifiedKFold,cross_val_score,KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression,LinearRegression
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score, root_mean_squared_error

## Dataset Load

In [2]:
df=pd.DataFrame(pd.read_csv(r"retail_pos_synthetic.csv"))

## First Look

In [3]:
df

,product_id,product_name,category,subcategory,brand,unit_of_measure,supplier_id,store_id,store_type,store_location,...,minimum_order_quantity,last_purchase_quantity,days_since_last_restock,day_of_week,week,month,season,holiday_or_event,future_demand,stockout_target
0,P0117,ValueMax Soft Drinks 117,Beverages,Soft Drinks,HomeEase,kg,S014,ST031,Convenience,Zone-4,...,5,31.0,15.0,Sunday,34,8,Monsoon,NaN,6,0
1,P0013,PrimeLine Oral Care 013,Personal Care,Oral Care,UrbanPlus,litre,S017,ST025,Convenience,Zone-5,...,25,40.0,7.0,Wednesday,16,4,Summer,NaN,5,0
2,P0430,PrimeLine Accessories 430,Electronics,Accessories,UrbanPlus,bottle,S008,ST004,Supermarket,Zone-7,...,10,22.0,5.0,Friday,22,6,Monsoon,NaN,7,0
3,P0035,ValueMax Laundry 035,Household,Laundry,ValueMax,litre,S008,ST037,Hypermarket,Zone-1,...,10,71.0,12.0,Tuesday,3,1,Winter,NaN,8,1
4,P0387,FreshMart Rice 387,Grocery,Rice,Nova,bottle,S028,ST027,Hypermarket,Zone-8,...,20,266.0,19.0,Thursday,4,1,Winter,NaN,24,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
999995,P0334,FreshMart Snacks 334,Grocery,Snacks,ValueMax,litre,S023,ST044,Supermarket,Zone-5,...,25,97.0,12.0,Monday,6,2,Winter,NaN,13,0
999996,P0480,ValueMax Deodorant 480,Personal Care,Deodorant,PrimeLine,litre,S030,ST005,Hypermarket,Zone-8,...,100,131.0,29.0,Tuesday,32,8,Monsoon,NaN,16,0
999997,P0144,ValueMax Flour 144,Grocery,Flour,Apex,litre,S014,ST045,Supermarket,Zone-5,...,100,100.0,NaN,Thursday,15,4,Summer,NaN,8,0
999998,P0375,Nova Paper 375,Household,Paper,DailyChoice,bottle,S010,ST038,Supermarket,Zone-7,...,10,41.0,18.0,Thursday,15,4,Summer,NaN,8,0


## Basic Inspection

In [4]:
df.head()

,product_id,product_name,category,subcategory,brand,unit_of_measure,supplier_id,store_id,store_type,store_location,...,minimum_order_quantity,last_purchase_quantity,days_since_last_restock,day_of_week,week,month,season,holiday_or_event,future_demand,stockout_target
0,P0117,ValueMax Soft Drinks 117,Beverages,Soft Drinks,HomeEase,kg,S014,ST031,Convenience,Zone-4,...,5,31.0,15.0,Sunday,34,8,Monsoon,NaN,6,0
1,P0013,PrimeLine Oral Care 013,Personal Care,Oral Care,UrbanPlus,litre,S017,ST025,Convenience,Zone-5,...,25,40.0,7.0,Wednesday,16,4,Summer,NaN,5,0
2,P0430,PrimeLine Accessories 430,Electronics,Accessories,UrbanPlus,bottle,S008,ST004,Supermarket,Zone-7,...,10,22.0,5.0,Friday,22,6,Monsoon,NaN,7,0
3,P0035,ValueMax Laundry 035,Household,Laundry,ValueMax,litre,S008,ST037,Hypermarket,Zone-1,...,10,71.0,12.0,Tuesday,3,1,Winter,NaN,8,1
4,P0387,FreshMart Rice 387,Grocery,Rice,Nova,bottle,S028,ST027,Hypermarket,Zone-8,...,20,266.0,19.0,Thursday,4,1,Winter,NaN,24,1


In [5]:
df.tail()

,product_id,product_name,category,subcategory,brand,unit_of_measure,supplier_id,store_id,store_type,store_location,...,minimum_order_quantity,last_purchase_quantity,days_since_last_restock,day_of_week,week,month,season,holiday_or_event,future_demand,stockout_target
999995,P0334,FreshMart Snacks 334,Grocery,Snacks,ValueMax,litre,S023,ST044,Supermarket,Zone-5,...,25,97.0,12.0,Monday,6,2,Winter,NaN,13,0
999996,P0480,ValueMax Deodorant 480,Personal Care,Deodorant,PrimeLine,litre,S030,ST005,Hypermarket,Zone-8,...,100,131.0,29.0,Tuesday,32,8,Monsoon,NaN,16,0
999997,P0144,ValueMax Flour 144,Grocery,Flour,Apex,litre,S014,ST045,Supermarket,Zone-5,...,100,100.0,NaN,Thursday,15,4,Summer,NaN,8,0
999998,P0375,Nova Paper 375,Household,Paper,DailyChoice,bottle,S010,ST038,Supermarket,Zone-7,...,10,41.0,18.0,Thursday,15,4,Summer,NaN,8,0
999999,P0166,PrimeLine Tea 166,Beverages,Tea,HomeEase,unit,S013,ST010,Supermarket,Zone-3,...,20,51.0,5.0,Saturday,51,12,Winter,Festival_Event,14,0


In [6]:
df.shape

(1000000, 40)

In [7]:
df.columns

Index(['product_id', 'product_name', 'category', 'subcategory', 'brand',
       'unit_of_measure', 'supplier_id', 'store_id', 'store_type',
       'store_location', 'city', 'region', 'date', 'quantity_sold',
       'sales_amount', 'number_of_transactions', 'average_selling_price',
       'opening_stock', 'closing_stock', 'current_stock', 'stock_received',
       'stock_adjustment', 'stockout_indicator', 'previous_stockout_count',
       'selling_price', 'purchase_price', 'discount_percentage',
       'promotion_active', 'promotion_type', 'supplier_lead_time',
       'minimum_order_quantity', 'last_purchase_quantity',
       'days_since_last_restock', 'day_of_week', 'week', 'month', 'season',
       'holiday_or_event', 'future_demand', 'stockout_target'],
      dtype='object')

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 40 columns):
 #   Column                   Non-Null Count    Dtype  
---  ------                   --------------    -----  
 0   product_id               1000000 non-null  object 
 1   product_name             1000000 non-null  object 
 2   category                 1000000 non-null  object 
 3   subcategory              1000000 non-null  object 
 4   brand                    1000000 non-null  object 
 5   unit_of_measure          1000000 non-null  object 
 6   supplier_id              1000000 non-null  object 
 7   store_id                 1000000 non-null  object 
 8   store_type               1000000 non-null  object 
 9   store_location           1000000 non-null  object 
 10  city                     1000000 non-null  object 
 11  region                   1000000 non-null  object 
 12  date                     1000000 non-null  object 
 13  quantity_sold            1000000 non-null  

In [9]:
df.describe()

,quantity_sold,sales_amount,number_of_transactions,average_selling_price,opening_stock,closing_stock,current_stock,stock_received,stock_adjustment,stockout_indicator,...,discount_percentage,promotion_active,supplier_lead_time,minimum_order_quantity,last_purchase_quantity,days_since_last_restock,week,month,future_demand,stockout_target
count,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,994091.000000,1000000.000000,...,987993.000000,1000000.000000,991990.000000,1000000.000000,987990.000000,989999.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000
mean,7.932199,886.719804,4.252881,109.157056,68.823961,78.991984,78.991984,18.013138,-0.001839,0.006714,...,2.446007,0.119660,4.497030,35.017880,104.438372,14.488238,24.945884,6.161008,11.695187,0.160000
std,6.843722,1156.116859,4.025301,78.217686,61.376925,83.641864,83.641864,51.846048,4.865511,0.081664,...,4.717498,0.324564,2.292493,32.421342,80.545415,8.657167,15.343979,3.521546,8.308532,0.366606
min,0.000000,0.000000,1.000000,8.590000,2.000000,0.000000,0.000000,0.000000,-10.000000,0.000000,...,0.000000,0.000000,1.000000,5.000000,5.000000,0.000000,1.000000,1.000000,0.000000,0.000000
25%,3.000000,242.220000,2.000000,55.440000,28.000000,27.000000,27.000000,0.000000,-2.000000,0.000000,...,0.230000,0.000000,2.000000,10.000000,50.000000,7.000000,11.000000,3.000000,6.000000,0.000000
50%,6.000000,528.105000,3.000000,89.720000,51.000000,54.000000,54.000000,0.000000,0.000000,0.000000,...,0.970000,0.000000,4.000000,25.000000,91.000000,14.000000,24.000000,6.000000,10.000000,0.000000
75%,10.000000,1092.450000,5.000000,136.670000,89.000000,102.000000,102.000000,0.000000,2.000000,0.000000,...,1.840000,0.000000,6.000000,50.000000,126.000000,22.000000,38.000000,9.000000,15.000000,0.000000
max,320.000000,49360.560000,90.000000,732.980000,1376.000000,1890.000000,1890.000000,1209.000000,10.000000,1.000000,...,40.000000,1.000000,8.000000,100.000000,2241.000000,29.000000,52.000000,12.000000,156.000000,1.000000


In [10]:
df.isnull().sum()

product_id                      0
product_name                    0
category                        0
subcategory                     0
brand                           0
unit_of_measure                 0
supplier_id                     0
store_id                        0
store_type                      0
store_location                  0
city                            0
region                          0
date                            0
quantity_sold                   0
sales_amount                    0
number_of_transactions          0
average_selling_price           0
opening_stock                   0
closing_stock                   0
current_stock                   0
stock_received                  0
stock_adjustment             5909
stockout_indicator              0
previous_stockout_count         0
selling_price                   0
purchase_price                  0
discount_percentage         12007
promotion_active                0
promotion_type             881532
supplier_lead_

In [11]:
df.duplicated()

0         False
1         False
2         False
3         False
4         False
          ...  
999995    False
999996    False
999997    False
999998    False
999999    False
Length: 1000000, dtype: bool

In [12]:
df.duplicated().sum()

0

In [13]:
df.dtypes

product_id                  object
product_name                object
category                    object
subcategory                 object
brand                       object
unit_of_measure             object
supplier_id                 object
store_id                    object
store_type                  object
store_location              object
city                        object
region                      object
date                        object
quantity_sold                int64
sales_amount               float64
number_of_transactions       int64
average_selling_price      float64
opening_stock                int64
closing_stock                int64
current_stock                int64
stock_received               int64
stock_adjustment           float64
stockout_indicator           int64
previous_stockout_count      int64
selling_price              float64
purchase_price             float64
discount_percentage        float64
promotion_active             int64
promotion_type      

In [14]:

df["category"].unique()

array(['Beverages', 'Personal Care', 'Electronics', 'Household',
       'Grocery'], dtype=object)

In [15]:
df['stockout_target'].value_counts()

stockout_target
0    840000
1    160000
Name: count, dtype: int64

## Feature Engineering

In [16]:
df['date'] = pd.to_datetime(df['date'])

In [17]:
df['day'] = df['date'].dt.day
df['day_of_week_num'] = df['date'].dt.dayofweek
df['quarter'] = df['date'].dt.quarter
df['is_weekend'] = (df['date'].dt.dayofweek >= 5).astype(int)

In [18]:
df['sales_per_transaction'] = (df['quantity_sold'] / df['number_of_transactions'].replace(0, np.nan))

df['revenue_per_unit'] = (df['sales_amount'] / df['quantity_sold'].replace(0, np.nan))

In [19]:
df['sales_per_transaction']

0         2.000000
1         3.000000
2         3.000000
3         2.500000
4         1.095238
            ...   
999995    1.250000
999996    1.571429
999997    1.000000
999998    2.000000
999999    1.142857
Name: sales_per_transaction, Length: 1000000, dtype: float64

In [20]:
df['revenue_per_unit']

0          94.847500
1          68.946667
2         119.880000
3          47.264000
4         195.360870
             ...    
999995    127.517000
999996     81.493636
999997     84.810000
999998     87.977500
999999     98.661250
Name: revenue_per_unit, Length: 1000000, dtype: float64

In [21]:
df['stock_consumption_ratio'] = (df['quantity_sold'] / df['opening_stock'].replace(0, np.nan))

df['stock_to_sales_ratio'] = (df['current_stock'] / df['quantity_sold'].replace(0, np.nan))

df['inventory_change'] = (df['closing_stock'] - df['opening_stock'])

df['stock_gap'] = ( df['opening_stock'] - df['current_stock'])

In [22]:
df['stock_consumption_ratio']

0         0.166667
1         0.058824
2         0.103448
3         0.138889
4         0.179688
            ...   
999995    0.138889
999996    0.132530
999997    0.285714
999998    0.083333
999999    0.145455
Name: stock_consumption_ratio, Length: 1000000, dtype: float64

In [23]:
df['stock_to_sales_ratio']

0          4.750000
1         16.333333
2         14.333333
3          5.200000
4          5.000000
            ...    
999995     6.200000
999996    19.363636
999997     1.666667
999998    13.500000
999999     4.625000
Name: stock_to_sales_ratio, Length: 1000000, dtype: float64

In [24]:
df['inventory_change']

0          -5
1          -2
2          14
3         -10
4         -13
         ... 
999995    -10
999996    130
999997    -11
999998      6
999999    -18
Name: inventory_change, Length: 1000000, dtype: int64

In [25]:
df['stock_gap']

0           5
1           2
2         -14
3          10
4          13
         ... 
999995     10
999996   -130
999997     11
999998     -6
999999     18
Name: stock_gap, Length: 1000000, dtype: int64

In [26]:
df['lead_time_stock_pressure'] = (df['supplier_lead_time'] * df['quantity_sold'])

df['restock_urgency'] = (df['current_stock'] /(df['supplier_lead_time'] + 1))

df['purchase_gap'] = (df['minimum_order_quantity'] - df['current_stock'])

In [27]:
df['lead_time_stock_pressure']

0         24.0
1          3.0
2          9.0
3          5.0
4         23.0
          ... 
999995    70.0
999996    88.0
999997    36.0
999998    24.0
999999    48.0
Name: lead_time_stock_pressure, Length: 1000000, dtype: float64

In [28]:
df['restock_urgency']

0          2.714286
1         24.500000
2         10.750000
3         13.000000
4         57.500000
            ...    
999995     7.750000
999996    23.666667
999997     1.428571
999998     7.714286
999999     5.285714
Name: restock_urgency, Length: 1000000, dtype: float64

In [29]:
df['purchase_gap']

0         -14
1         -24
2         -33
3         -16
4         -95
         ... 
999995    -37
999996   -113
999997     90
999998    -44
999999    -17
Name: purchase_gap, Length: 1000000, dtype: int64

In [30]:
df['profit_per_unit'] = (df['selling_price'] - df['purchase_price'])

df['profit_margin_pct'] = ((df['selling_price'] - df['purchase_price'])/ df['selling_price'].replace(0, np.nan)) * 100

df['discount_amount'] = (df['selling_price'] * df['discount_percentage'] / 100)

In [31]:
df['profit_per_unit']

0         15.75
1         12.64
2         23.66
3          1.52
4         54.91
          ...  
999995    30.45
999996    25.82
999997    23.39
999998    17.48
999999    18.04
Name: profit_per_unit, Length: 1000000, dtype: float64

In [32]:
df['profit_margin_pct']

0         16.605166
1         18.332125
2         19.736403
3          3.216251
4         28.107084
            ...    
999995    23.878607
999996    31.684869
999997    27.579295
999998    19.868152
999999    18.285019
Name: profit_margin_pct, Length: 1000000, dtype: float64

In [33]:
df['discount_amount']

0         1.043350
1         0.000000
2         4.543452
3         8.728922
4         0.820512
            ...   
999995    1.262448
999996    2.094293
999997    0.746328
999998    1.152538
999999    1.233250
Name: discount_amount, Length: 1000000, dtype: float64

In [34]:
df['promotion_sales_effect'] = (df['quantity_sold'] * df['promotion_active'])

In [35]:
df['promotion_sales_effect']

0         0
1         0
2         0
3         5
4         0
         ..
999995    0
999996    0
999997    0
999998    0
999999    0
Name: promotion_sales_effect, Length: 1000000, dtype: int64

In [36]:
df['has_previous_stockout'] = (df['previous_stockout_count'] > 0).astype(int)

In [37]:
df['has_previous_stockout']

0         0
1         0
2         0
3         0
4         1
         ..
999995    0
999996    0
999997    0
999998    0
999999    0
Name: has_previous_stockout, Length: 1000000, dtype: int32

## Target aur Features Separate

In [38]:
X = df.drop(["stockout_target", "future_demand", "date"],axis=1).copy()

y = df["stockout_target"].copy()

In [39]:
print(X.shape)

(1000000, 55)


In [40]:
print(y.shape)

(1000000,)


## Train - Test Split

In [41]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=42,stratify=y)

In [42]:
print(X_train.shape)
print(X_test.shape)

(800000, 55)
(200000, 55)


In [43]:
print(y_train.shape)
print(y_test.shape)

(800000,)
(200000,)


In [44]:
id_columns = ['product_id','product_name','supplier_id','store_id']

X_train = X_train.drop(columns=id_columns)
X_test = X_test.drop(columns=id_columns)

In [45]:
print("X_train Shape:", X_train.shape)
print("X_test Shape :", X_test.shape)

X_train Shape: (800000, 51)
X_test Shape : (200000, 51)


## Numerical & Categorical Columns

In [78]:
numerical_cols=Pipeline([("imputer",SimpleImputer(strategy="median")),("scaler",StandardScaler())])

In [79]:
numerical_cols

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())])

In [86]:
categorical_cols=Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),("encoder",OneHotEncoder(sparse_output=False))])

In [87]:
categorical_cols

Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')),
                ('encoder', OneHotEncoder(sparse_output=False))])

In [89]:
complete_pipeline = ColumnTransformer([("number",numerical_cols,X_train.select_dtypes(include=np.number).columns.tolist()),
                                       ("category",categorical_cols,X_train.select_dtypes(exclude=np.number).columns.tolist())],verbose_feature_names_out=False,verbose=True).set_output(transform="pandas")

In [90]:
complete_pipeline

ColumnTransformer(transformers=[('number',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['quantity_sold', 'sales_amount',
                                  'number_of_transactions',
                                  'average_selling_price', 'opening_stock',
                                  'closing_stock', 'current_stock',
                                  'stock_received', 'stock_adjustment',
                                  'stockout_indicator',
                                  'previous_stockout_count', 'sell...
                                  'stock_to_sales_ratio', 'inventory_change', ...]),
                                ('category',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(sparse_output=False))]),
                                 ['category', 'subcategory', 'brand',
                                  'unit_of_measure', 'store_type',
                                  'store_location', 'city', 'region',
                                  'promotion_type', 'day_of_week', 'season',
                                  'holiday_or_event'])],
                  verbose=True, verbose_feature_names_out=False)

## Model Pipeline

In [91]:
model_pipeline = Pipeline([("preprocessing", complete_pipeline),("model", LogisticRegression(max_iter=1000,random_state=42))])

In [92]:
model_pipeline

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('number',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['quantity_sold',
                                                   'sales_amount',
                                                   'number_of_transactions',
                                                   'average_selling_price',
                                                   'opening_stock',
                                                   'closing_stock',
                                                   'current_stock',
                                                   'stock_received',
                                                   'stock_adjustment',
                                                   'stockout_indicator...
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(sparse_output=False))]),
                                                  ['category', 'subcategory',
                                                   'brand', 'unit_of_measure',
                                                   'store_type',
                                                   'store_location', 'city',
                                                   'region', 'promotion_type',
                                                   'day_of_week', 'season',
                                                   'holiday_or_event'])],
                                   verbose=True,
                                   verbose_feature_names_out=False)),
                ('model', LogisticRegression(max_iter=1000, random_state=42))])

In [95]:
model_pipeline.fit(X_train, y_train)

[ColumnTransformer] ........ (1 of 2) Processing number, total=   9.2s
[ColumnTransformer] ...... (2 of 2) Processing category, total=   6.1s


Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('number',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['quantity_sold',
                                                   'sales_amount',
                                                   'number_of_transactions',
                                                   'average_selling_price',
                                                   'opening_stock',
                                                   'closing_stock',
                                                   'current_stock',
                                                   'stock_received',
                                                   'stock_adjustment',
                                                   'stockout_indicator...
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(sparse_output=False))]),
                                                  ['category', 'subcategory',
                                                   'brand', 'unit_of_measure',
                                                   'store_type',
                                                   'store_location', 'city',
                                                   'region', 'promotion_type',
                                                   'day_of_week', 'season',
                                                   'holiday_or_event'])],
                                   verbose=True,
                                   verbose_feature_names_out=False)),
                ('model', LogisticRegression(max_iter=1000, random_state=42))])

## Accuracy

In [96]:
train_accuracy = model_pipeline.score(X_train, y_train)

print("Training Accuracy:", train_accuracy)

Training Accuracy: 0.88529875


## Cross-Validation

In [97]:
cv = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

In [98]:
cv_accuracy = cross_val_score(model_pipeline,X_train,y_train,cv=cv,scoring="accuracy",n_jobs=-1)

In [99]:
cv_accuracy

array([0.884725  , 0.8859625 , 0.88433125, 0.88505625, 0.885825  ])

In [100]:
cv_accuracy.mean()

0.8851799999999999

## Predication

In [101]:
y_pred = model_pipeline.predict(X_test)

In [102]:
y_pred

array([0, 0, 1, ..., 0, 0, 1], dtype=int64)

##  Future Demand Prediction

## Target aur Features Separate

In [103]:
X_reg = df.drop(["future_demand", "stockout_target", "date"],axis=1).copy()

y_reg = df["future_demand"].copy()

In [104]:
print(X_reg.shape)

(1000000, 55)


In [105]:
print(y_reg.shape)

(1000000,)


In [106]:
y_reg.describe()

count    1000000.000000
mean          11.695187
std            8.308532
min            0.000000
25%            6.000000
50%           10.000000
75%           15.000000
max          156.000000
Name: future_demand, dtype: float64

In [107]:
id_columns = ["product_id","product_name","supplier_id","store_id"]
X_reg = X_reg.drop(columns=id_columns)

In [108]:
print("X_reg Shape:", X_reg.shape)

X_reg Shape: (1000000, 51)


In [109]:
X_reg.columns

Index(['category', 'subcategory', 'brand', 'unit_of_measure', 'store_type',
       'store_location', 'city', 'region', 'quantity_sold', 'sales_amount',
       'number_of_transactions', 'average_selling_price', 'opening_stock',
       'closing_stock', 'current_stock', 'stock_received', 'stock_adjustment',
       'stockout_indicator', 'previous_stockout_count', 'selling_price',
       'purchase_price', 'discount_percentage', 'promotion_active',
       'promotion_type', 'supplier_lead_time', 'minimum_order_quantity',
       'last_purchase_quantity', 'days_since_last_restock', 'day_of_week',
       'week', 'month', 'season', 'holiday_or_event', 'day', 'day_of_week_num',
       'quarter', 'is_weekend', 'sales_per_transaction', 'revenue_per_unit',
       'stock_consumption_ratio', 'stock_to_sales_ratio', 'inventory_change',
       'stock_gap', 'lead_time_stock_pressure', 'restock_urgency',
       'purchase_gap', 'profit_per_unit', 'profit_margin_pct',
       'discount_amount', 'promotion_sal

## Train - Test Split

In [110]:
X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(X_reg,y_reg,test_size=0.20,random_state=42)

In [111]:
print(X_reg.shape)
print(X_reg_test.shape)

(1000000, 51)
(200000, 51)


In [112]:
print(y_reg.shape)
print(y_reg_test.shape)

(1000000,)
(200000,)


## Numerical & Categorical Columns

In [113]:
reg_numeric_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median")),("scaler", StandardScaler())])

In [114]:
reg_numeric_pipeline

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())])

In [115]:
reg_categorical_pipeline=Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),("encoder",OneHotEncoder(sparse_output=False))])

In [116]:
reg_categorical_pipeline

Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')),
                ('encoder', OneHotEncoder(sparse_output=False))])

In [117]:
reg_complete_pipeline= ColumnTransformer([("number",reg_numeric_pipeline,X_train.select_dtypes(include=np.number).columns.tolist()),
                                       ("category",reg_categorical_pipeline,X_train.select_dtypes(exclude=np.number).columns.tolist())],verbose_feature_names_out=False,verbose=True).set_output(transform="pandas")

In [118]:
reg_complete_pipeline

ColumnTransformer(transformers=[('number',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['quantity_sold', 'sales_amount',
                                  'number_of_transactions',
                                  'average_selling_price', 'opening_stock',
                                  'closing_stock', 'current_stock',
                                  'stock_received', 'stock_adjustment',
                                  'stockout_indicator',
                                  'previous_stockout_count', 'sell...
                                  'stock_to_sales_ratio', 'inventory_change', ...]),
                                ('category',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(sparse_output=False))]),
                                 ['category', 'subcategory', 'brand',
                                  'unit_of_measure', 'store_type',
                                  'store_location', 'city', 'region',
                                  'promotion_type', 'day_of_week', 'season',
                                  'holiday_or_event'])],
                  verbose=True, verbose_feature_names_out=False)

## Model Pipeline

In [119]:
reg_model_pipeline = Pipeline([("preprocessing", reg_complete_pipeline),("model", LinearRegression())])

In [120]:
reg_model_pipeline

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('number',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['quantity_sold',
                                                   'sales_amount',
                                                   'number_of_transactions',
                                                   'average_selling_price',
                                                   'opening_stock',
                                                   'closing_stock',
                                                   'current_stock',
                                                   'stock_received',
                                                   'stock_adjustment',
                                                   'stockout_indicator...
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(sparse_output=False))]),
                                                  ['category', 'subcategory',
                                                   'brand', 'unit_of_measure',
                                                   'store_type',
                                                   'store_location', 'city',
                                                   'region', 'promotion_type',
                                                   'day_of_week', 'season',
                                                   'holiday_or_event'])],
                                   verbose=True,
                                   verbose_feature_names_out=False)),
                ('model', LinearRegression())])

In [121]:
reg_model_pipeline.fit(X_reg_train,y_reg_train)

[ColumnTransformer] ........ (1 of 2) Processing number, total=   8.1s
[ColumnTransformer] ...... (2 of 2) Processing category, total=   6.5s


Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('number',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['quantity_sold',
                                                   'sales_amount',
                                                   'number_of_transactions',
                                                   'average_selling_price',
                                                   'opening_stock',
                                                   'closing_stock',
                                                   'current_stock',
                                                   'stock_received',
                                                   'stock_adjustment',
                                                   'stockout_indicator...
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(sparse_output=False))]),
                                                  ['category', 'subcategory',
                                                   'brand', 'unit_of_measure',
                                                   'store_type',
                                                   'store_location', 'city',
                                                   'region', 'promotion_type',
                                                   'day_of_week', 'season',
                                                   'holiday_or_event'])],
                                   verbose=True,
                                   verbose_feature_names_out=False)),
                ('model', LinearRegression())])

## Predication

In [131]:
y_reg_pred = reg_model_pipeline.predict(X_reg_test)

In [132]:
y_reg_pred

array([17.71359253, 11.29928589, 15.83816528, ...,  3.98236084,
        2.53192139, 13.69671631])

## R2

In [138]:
r2 = r2_score(y_reg_test,y_reg_pred)

In [139]:
print(r2)

0.9342749628781333


## Cross-Validation

In [140]:
cv_reg = KFold(n_splits=5,shuffle=True,random_state=42)

In [141]:
cv_mae = -cross_val_score(reg_model_pipeline,X_reg_train,y_reg_train,cv=cv_reg,n_jobs=-1)

In [142]:
cv_mae.mean()

127724093.06728777

In [143]:
cv_r2 = cross_val_score(reg_model_pipeline,X_reg_train,y_reg_train,cv=cv_reg,scoring="r2",n_jobs=-1)

In [144]:
cv_r2

array([ 9.34727846e-01,  9.34860016e-01,  9.34825450e-01, -6.38620469e+08,
        9.33439123e-01])

## Intelligent Replenishment Recommendation

In [450]:
recommendation_df = df.loc[X_test.index].copy()


In [451]:
recommendation_df["stockout_probability"] = y_prob


In [452]:
X_reg_test_model3 = X_reg.loc[X_test.index]


In [453]:
recommendation_df["predicted_future_demand"] = (
    reg_model_pipeline.predict(X_reg_test_model3)
)

In [458]:
recommendation_df

,product_id,product_name,category,subcategory,brand,unit_of_measure,supplier_id,store_id,store_type,store_location,...,profit_per_unit,profit_margin_pct,discount_amount,promotion_sales_effect,has_previous_stockout,stockout_probability,predicted_future_demand,stockout_risk,demand_gap,reorder_quantity
722226,P0480,ValueMax Deodorant 480,Personal Care,Deodorant,PrimeLine,litre,S030,ST023,Hypermarket,Zone-8,...,19.06,25.505152,0.291447,0,0,0.015788,12.395294,LOW,-63.604706,-63.604706
354952,P0305,FreshMart Chargers 305,Electronics,Chargers,Apex,bottle,S015,ST047,Hypermarket,Zone-2,...,33.02,27.664209,0.226784,0,0,0.052048,10.920013,LOW,-53.079987,-53.079987
866721,P0248,UrbanPlus Chargers 248,Electronics,Chargers,FreshMart,litre,S027,ST017,Convenience,Zone-3,...,18.67,29.781464,0.112842,0,1,0.525542,4.866150,MEDIUM,-8.133850,-8.133850
213927,P0456,FreshMart Juice 456,Beverages,Juice,HomeEase,unit,S022,ST029,Supermarket,Zone-1,...,8.60,23.312551,0.018445,0,0,0.080478,21.950409,LOW,-22.049591,-22.049591
178646,P0348,Apex Pulses 348,Grocery,Pulses,UrbanPlus,unit,S020,ST046,Supermarket,Zone-1,...,44.64,24.673889,1.700648,0,0,0.018083,23.552887,LOW,-382.447113,-382.447113
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
265775,P0185,Nova Batteries 185,Electronics,Batteries,UrbanPlus,unit,S023,ST031,Convenience,Zone-4,...,14.95,19.032463,2.144415,0,1,0.222274,4.081238,LOW,-46.918762,-46.918762
535049,P0224,DailyChoice Water 224,Beverages,Water,PrimeLine,kg,S020,ST018,Supermarket,Zone-3,...,14.42,23.985363,0.955908,0,0,0.025889,22.182892,LOW,-86.817108,-86.817108
867679,P0290,Nova Storage 290,Household,Storage,UrbanPlus,pack,S024,ST008,Convenience,Zone-5,...,17.77,22.533604,0.654538,0,0,0.039608,7.302704,LOW,-46.697296,-46.697296
784680,P0349,Apex Small Gadgets 349,Electronics,Small Gadgets,DailyChoice,litre,S018,ST048,Express,Zone-5,...,11.40,14.913658,12.039300,13,0,0.138659,20.313934,LOW,-153.686066,-153.686066


In [459]:
recommendation_df["stockout_risk"] = pd.cut(recommendation_df["stockout_probability"],bins=[-np.inf, 0.30, 0.70, np.inf],labels=["LOW", "MEDIUM", "HIGH"])

In [460]:
recommendation_df["stockout_risk"]

722226       LOW
354952       LOW
866721    MEDIUM
213927       LOW
178646       LOW
           ...  
265775       LOW
535049       LOW
867679       LOW
784680       LOW
164905      HIGH
Name: stockout_risk, Length: 200000, dtype: category
Categories (3, object): ['LOW' < 'MEDIUM' < 'HIGH']

In [461]:
recommendation_df["demand_gap"] = (recommendation_df["predicted_future_demand"]- recommendation_df["current_stock"])

In [462]:
recommendation_df["demand_gap"]

722226    -63.604706
354952    -53.079987
866721     -8.133850
213927    -22.049591
178646   -382.447113
             ...    
265775    -46.918762
535049    -86.817108
867679    -46.697296
784680   -153.686066
164905      0.597626
Name: demand_gap, Length: 200000, dtype: float64

In [463]:
recommendation_df["reorder_quantity"] = (recommendation_df["demand_gap"])

In [465]:
recommendation_df["recommended_order_quantity"] = np.where(
    recommendation_df["reorder_quantity"] > 0,np.maximum(recommendation_df["reorder_quantity"],recommendation_df["minimum_order_quantity"]),0)

In [466]:
recommendation_df["recommended_order_quantity"]

722226      0.0
354952      0.0
866721      0.0
213927      0.0
178646      0.0
          ...  
265775      0.0
535049      0.0
867679      0.0
784680      0.0
164905    100.0
Name: recommended_order_quantity, Length: 200000, dtype: float64

In [467]:
recommendation_df["reorder_decision"] = np.where( recommendation_df["recommended_order_quantity"] > 0,"REORDER","NO REORDER")

In [468]:
recommendation_df["reorder_decision"]

722226    NO REORDER
354952    NO REORDER
866721    NO REORDER
213927    NO REORDER
178646    NO REORDER
             ...    
265775    NO REORDER
535049    NO REORDER
867679    NO REORDER
784680    NO REORDER
164905       REORDER
Name: reorder_decision, Length: 200000, dtype: object

In [3]:
recommendation_df["priority"] = np.select(
    [
        (
            (recommendation_df["stockout_risk"] == "HIGH") &
            (recommendation_df["recommended_order_quantity"] > 0)
        ),
        (
            (recommendation_df["stockout_risk"] == "MEDIUM") &
            (recommendation_df["recommended_order_quantity"] > 0)
        ),
        (
            (recommendation_df["stockout_risk"] == "LOW") &
            (recommendation_df["recommended_order_quantity"] > 0)
        )
    ],
    [
        "HIGH",
        "MEDIUM",
        "LOW"
    ],
    default="NO ACTION"
)

NameError: name 'recommendation_df' is not defined

In [4]:
final_recommendation = recommendation_df[
    [
        "product_name",
        "category",
        "store_location",

        "current_stock",
        "predicted_future_demand",
        "demand_gap",

        "stockout_probability",
        "stockout_risk",

        "supplier_lead_time",
        "recommended_order_quantity",

        "reorder_decision",
        "priority"
    ]
].copy()

display(final_recommendation.head(20))

NameError: name 'recommendation_df' is not defined